# Data

In [1]:
import json

with open('data/result.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

In [2]:
import pandas as pd

df = pd.DataFrame(data['messages'])

In [3]:
from utils import preprocess_df

df = preprocess_df(df)

Загружено сообщений для анализа: 32738


# Pipeline

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
import umap

reducer = umap.UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

In [6]:
from sklearn.cluster import DBSCAN

clusterer = DBSCAN(eps=0.001, min_samples=3)

In [7]:
from bertopic import BERTopic

topic_model = BERTopic(
    embedding_model=model,
    hdbscan_model=clusterer,
    umap_model=reducer,
    verbose=True
)

topics, probs = topic_model.fit_transform(df['clean_text'].tolist())

2026-03-29 16:03:06,992 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/1024 [00:00<?, ?it/s]

2026-03-29 16:05:17,786 - BERTopic - Embedding - Completed ✓
2026-03-29 16:05:17,787 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-29 16:06:13,757 - BERTopic - Dimensionality - Completed ✓
2026-03-29 16:06:13,759 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-29 16:06:14,180 - BERTopic - Cluster - Completed ✓
2026-03-29 16:06:14,190 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-29 16:06:14,519 - BERTopic - Representation - Completed ✓


# Analysis

In [8]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,31407,-1_не_на_ты_бля,"[не, на, ты, бля, это, что, ну, как, он, да]","[Или че там было, Как то, Ну не надо так]"
1,0,51,0_gyrozeppeli2_попали_рест_открыла,"[gyrozeppeli2, попали, рест, открыла, тусит, п...","[@Gyrozeppeli2, @Gyrozeppeli2, @Gyrozeppeli2]"
2,1,44,1_esktpnk_киря_объяснись_сдавай,"[esktpnk, киря, объяснись, сдавай, подскажи, о...","[@esktpnk, @esktpnk, @esktpnk]"
3,2,38,2_любимую_песню_найти_улюблену,"[любимую, песню, найти, улюблену, знайти, пісн...","[🔍 Найти любимую песню!, 🔍 Найти любимую песню..."
4,3,34,3_ахахахах_аххахахах__,"[ахахахах, аххахахах, , , , , , , , ]","[Ахахахах, Ахахахах, Ахахахах]"
...,...,...,...,...,...
171,170,3,170_кимпинь_кимпиньтяоо_кимпиньтю_кимпиньтяо,"[кимпинь, кимпиньтяоо, кимпиньтю, кимпиньтяо, ...","[кимпиньтяо💀, Кимпиньтяоо, Кимпинь кимпиньтю]"
172,171,3,171_ненавижу_энчантрес_пугачеву_всех,"[ненавижу, энчантрес, пугачеву, всех, бля, , ,...","[Я ненавижу Пугачеву, я бля всех ненавижу, Я н..."
173,172,3,172_красавчик_красавца__,"[красавчик, красавца, , , , , , , , ]","[Красавца, Красавчик, Красавчик]"
174,173,3,173_правда_это__,"[правда, это, , , , , , , , ]","[Это правда, Это правда, Это правда]"


In [9]:
topic_model.get_topic_info().describe()

,Topic,Count
count,176.000000,176.000000
mean,86.500000,186.011364
std,50.950957,2366.830116
min,-1.000000,3.000000
25%,42.750000,3.000000
50%,86.500000,5.000000
75%,130.250000,8.250000
max,174.000000,31407.000000
